In [1]:
import zipfile
import os
import pandas as pd

In [2]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [3]:
zip_path = "dataset.zip"      # uploaded file name
extract_path = "fraud_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP file extracted successfully!")

ZIP file extracted successfully!


In [4]:
os.listdir(extract_path + "/data")

['2018-09-17.pkl',
 '2018-06-15.pkl',
 '2018-08-26.pkl',
 '2018-05-18.pkl',
 '2018-04-23.pkl',
 '2018-08-29.pkl',
 '2018-09-03.pkl',
 '2018-04-06.pkl',
 '2018-05-11.pkl',
 '2018-08-27.pkl',
 '2018-09-27.pkl',
 '2018-08-12.pkl',
 '2018-07-04.pkl',
 '2018-08-11.pkl',
 '2018-04-20.pkl',
 '2018-08-21.pkl',
 '2018-09-09.pkl',
 '2018-06-10.pkl',
 '2018-07-13.pkl',
 '2018-06-01.pkl',
 '2018-06-14.pkl',
 '2018-06-05.pkl',
 '2018-04-08.pkl',
 '2018-07-19.pkl',
 '2018-06-08.pkl',
 '2018-08-08.pkl',
 '2018-05-21.pkl',
 '2018-08-15.pkl',
 '2018-09-24.pkl',
 '2018-07-14.pkl',
 '2018-05-12.pkl',
 '2018-07-21.pkl',
 '2018-08-01.pkl',
 '2018-04-26.pkl',
 '2018-09-26.pkl',
 '2018-07-28.pkl',
 '2018-08-28.pkl',
 '2018-08-25.pkl',
 '2018-05-03.pkl',
 '2018-04-25.pkl',
 '2018-05-14.pkl',
 '2018-07-11.pkl',
 '2018-05-30.pkl',
 '2018-04-04.pkl',
 '2018-04-03.pkl',
 '2018-06-09.pkl',
 '2018-08-14.pkl',
 '2018-09-04.pkl',
 '2018-06-26.pkl',
 '2018-06-04.pkl',
 '2018-07-01.pkl',
 '2018-07-25.pkl',
 '2018-04-07

In [5]:
base_path = "fraud_dataset/data"

dfs = []
for file in sorted(os.listdir(base_path)):
    if file.endswith(".pkl"):
        dfs.append(pd.read_pickle(os.path.join(base_path, file)))

df = pd.concat(dfs, ignore_index=True)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1754155, 9)


,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0


In [6]:
#base_path = extract_path + "/data"

#all_dfs = []

#for file in sorted(os.listdir(base_path)):
#    if file.endswith(".pkl"):
#        file_path = os.path.join(base_path, file)
#        df_day = pd.read_pickle(file_path)
#        all_dfs.append(df_day)

# Combine all daily data into one DataFrame
#df = pd.concat(all_dfs, ignore_index=True)

#print("Final dataset shape:", df.shape)
#df.head()

In [7]:
df.columns

Index(['TRANSACTION_ID', 'TX_DATETIME', 'CUSTOMER_ID', 'TERMINAL_ID',
       'TX_AMOUNT', 'TX_TIME_SECONDS', 'TX_TIME_DAYS', 'TX_FRAUD',
       'TX_FRAUD_SCENARIO'],
      dtype='object')

In [8]:
df.info()
df['TX_FRAUD'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1754155 entries, 0 to 1754154
Data columns (total 9 columns):
 #   Column             Dtype         
---  ------             -----         
 0   TRANSACTION_ID     int64         
 1   TX_DATETIME        datetime64[ns]
 2   CUSTOMER_ID        object        
 3   TERMINAL_ID        object        
 4   TX_AMOUNT          float64       
 5   TX_TIME_SECONDS    object        
 6   TX_TIME_DAYS       object        
 7   TX_FRAUD           int64         
 8   TX_FRAUD_SCENARIO  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(4)
memory usage: 120.4+ MB


,count
TX_FRAUD,
0,1739474
1,14681


In [9]:
from sklearn.preprocessing import LabelEncoder

In [10]:
# Drop non-informative column, ignore if already dropped
df.drop("TRANSACTION_ID", axis=1, inplace=True, errors='ignore')

In [11]:
# Convert datetime and extract features if 'TX_DATETIME' column exists
if "TX_DATETIME" in df.columns:
    # Ensure TX_DATETIME is datetime type (though it might already be)
    df["TX_DATETIME"] = pd.to_datetime(df["TX_DATETIME"])
    df["TX_HOUR"] = df["TX_DATETIME"].dt.hour
    df["TX_DAY"] = df["TX_DATETIME"].dt.day
    df["TX_DAY_OF_WEEK"] = df["TX_DATETIME"].dt.dayofweek
    # Drop TX_DATETIME after extracting features, ignore if already dropped (from prior successful run)
    df.drop("TX_DATETIME", axis=1, inplace=True, errors='ignore')

In [12]:
# Encode categorical IDs if they are still 'object' type
le = LabelEncoder()
if df["CUSTOMER_ID"].dtype == 'object':
    df["CUSTOMER_ID"] = le.fit_transform(df["CUSTOMER_ID"])
if df["TERMINAL_ID"].dtype == 'object':
    df["TERMINAL_ID"] = le.fit_transform(df["TERMINAL_ID"])

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X = df.drop("TX_FRAUD", axis=1)
y = df["TX_FRAUD"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=42)

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [17]:
rf = RandomForestClassifier(    random_state=42,    class_weight="balanced"
)

In [18]:
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [19]:
y_pred = rf.predict(X_test)

In [20]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    347895
           1       1.00      1.00      1.00      2936

    accuracy                           1.00    350831
   macro avg       1.00      1.00      1.00    350831
weighted avg       1.00      1.00      1.00    350831



The Random Forest model achieved an accuracy of 100%.
This is attributed to the simulated nature of the dataset, where fraud labels were generated using strong deterministic rules.
Since these patterns are easily separable, the model was able to perfectly classify transactions.
However, accuracy alone is insufficient for fraud detection, and metrics such as recall, precision, and F1-score were also evaluated.

USING BEST PARAMETERS

In [21]:
#from sklearn.model_selection import GridSearchCV

In [22]:
# param_grid = {
#     "n_estimators": [100, 200],
#     "max_depth": [None, 15, 25],
#     "min_samples_split": [2, 5],
#     "min_samples_leaf": [1, 2],
#     "class_weight": ["balanced"]
# }

In [23]:
#rf1 = RandomForestClassifier(random_state=42)

In [24]:
# grid = GridSearchCV(
#     rf1,
#     param_grid,
#     cv=3,
#     scoring="f1",
#     n_jobs=-1,
#     verbose=2
# )

In [25]:
#grid.fit(X_train, y_train)

In [26]:
#best_rf = grid.best_estimator_

In [27]:
#y_pred_best = best_rf.predict(X_test)

In [28]:
#print("Best Parameters:", grid.best_params_)

In [29]:
#print("Tuned Accuracy:", accuracy_score(y_test, y_pred_best))
#print(classification_report(y_test, y_pred_best))

In [30]:
#rf2 = RandomForestClassifier(random_state=42)

In [31]:
# grid1 = GridSearchCV(
#     rf2,
#     param_grid,
#     cv=3,
#     scoring="recall",
#     n_jobs=-1,
#     verbose=2
# )

In [32]:
#grid1.fit(X_train, y_train)

In [33]:
#best_rf2 = grid1.best_estimator_

In [34]:
#y_pred_best1 = best_rf2.predict(X_test)

In [35]:
#print("Best Parameters:", grid1.best_params_)

In [36]:
#print("Tuned Accuracy:", accuracy_score(y_test, y_pred_best1))
#print(classification_report(y_test, y_pred_best1))